# 9. Interdependence analysis over time

This notebook creates one analysis-level dataset and uses it to make two figures.


## 1. Create the analysis-level dataset

Each row in `analysis_data` represents one paper, one data-user country, and one standardized data holder. Holder links union the paper-level standardized holder field with holders explicitly inferred from the reviewed resource–holder dictionary; duplicate paper–country–holder rows are removed. The upstream field currently already contains the inferred links, so this explicit union verifies the rule without inflating counts.


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

# Import the packages used to prepare the data and draw the figures.
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Set the input file, output locations, analysis period, and countries shown in section 3.
INPUT_FILE = DATA_ROOT / "interdependence_paper_supplementary_data_new.xlsx"
ANALYSIS_DATA_FILE = INPUT_FILE.parent / "processed_data" / "interdependence_analysis_over_time_full_through_2024.csv"
FIGURE_DIR = DATA_ROOT / "visualizations"
FIGURE_DIR.mkdir(exist_ok=True)

END_YEAR = 2024
SELECTED_COUNTRIES = {
    "US": "United States",
    "CN": "China",
    "DE": "Germany",
    "JP": "Japan",
    "BR": "Brazil",
}

# Treat every EU-hosted institution as a national-public equivalent.
EU_MEMBER_COUNTRY_CODES = {
    "AT", "BE", "BG", "HR", "CY", "CZ", "DE", "DK",
    "EE", "ES", "FI", "FR", "GR", "HU", "IE", "IT",
    "LT", "LU", "LV", "MT", "NL", "PL", "PT", "RO",
    "SE", "SI", "SK",
}

# The main figures combine all external national-public sources.
CATEGORY_ORDER = [
    "Domestic public",
    "External public",
    "International public",
    "Research",
    "Private",
    "NGO-led",
    "Hybrid",
]

# The differentiated figures separate US and other external national-public sources.
DIFFERENTIATED_CATEGORY_ORDER = [
    "Domestic public",
    "External US public",
    "External other public",
    "International public",
    "Research",
    "Private",
    "NGO-led",
    "Hybrid",
]

# Categories used in the five country panels.
COUNTRY_COMPARISON_ORDER = [
    "Domestic data use",
    "External data use",
    "International public",
    "Private",
    "NGO",
]
COUNTRY_COMPARISON_DIFFERENTIATED_ORDER = [
    "Domestic data use",
    "External US data use",
    "External other data use",
    "International public",
    "Private",
    "NGO",
]

CATEGORY_COLORS = {
    "Domestic public": "#2f6f9f",
    "Domestic data use": "#1f77b4",
    "External data use": "#ff7f0e",
    "External US data use": "#d62728",
    "External other data use": "#ff7f0e",
    "External public": "#d95f02",
    "External US public": "#b2182b",
    "External other public": "#ef8a62",
    "International public": "#2ca02c",
    "Research": "#3f8f5f",
    "Private": "#7f7f7f",
    "NGO-led": "#c27c2c",
    "NGO": "#9467bd",
    "Hybrid": "#d4a72c",
}


In [ ]:
# Identify EU institutions that function as national-public equivalents here.
def is_eu_public_institution(row):
    return row["holder_country"] == "EU"


# Test whether a public or research holder is domestic to the data user.
def is_domestic_holder(row):
    if is_eu_public_institution(row):
        return row["user_country"] in EU_MEMBER_COUNTRY_CODES
    return row["user_country"] == row["holder_country"]


# Assign the differentiated analytical category shown in the additional figures.
def assign_differentiated_category(row):
    if row["holder_type"] == "national public" or is_eu_public_institution(row):
        if is_domestic_holder(row):
            return "Domestic public"
        if row["holder_country"] == "US":
            return "External US public"
        return "External other public"

    return {
        "international public": "International public",
        "research": "Research",
        "private": "Private",
        "ngo-led": "NGO-led",
        "hybrid": "Hybrid",
    }.get(row["holder_type"], "Other / unassigned")


# Combine national-public and research sources into domestic or external use.
def assign_country_comparison_category(row, differentiate_external=False):
    if (
        row["holder_type"] in ["national public", "research"]
        or is_eu_public_institution(row)
    ):
        if is_domestic_holder(row):
            return "Domestic data use"
        if differentiate_external and row["holder_country"] == "US":
            return "External US data use"
        if differentiate_external:
            return "External other data use"
        return "External data use"

    return {
        "international public": "International public",
        "private": "Private",
        "ngo-led": "NGO",
    }.get(row["holder_type"])


# Save each figure as both SVG and PNG.
def save_figure(figure, filename):
    filename = f"{filename}_through_2024"
    figure.savefig(FIGURE_DIR / f"{filename}.svg", bbox_inches="tight")
    figure.savefig(FIGURE_DIR / f"{filename}.png", dpi=300, bbox_inches="tight")


In [ ]:
# Load the paper data, reviewed resource–holder dictionary, and holder metadata.
papers = pd.read_excel(INPUT_FILE, sheet_name="paper_source_data")
source_metadata = pd.read_excel(INPUT_FILE, sheet_name="standardized_sources")
visualization_metadata = pd.read_excel(INPUT_FILE, sheet_name="visualization_names")

# Convert missing values to empty text and remove extra spaces.
def clean_text(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).replace("\xa0", " ").split())


# Split a semicolon-separated cell and keep each non-empty name once.
def split_names(value):
    names = []
    for part in clean_text(value).split(";"):
        name = clean_text(part)
        if name and name not in names:
            names.append(name)
    return names


# Return the first non-empty metadata value in a group.
def first_text(values):
    for value in values:
        value = clean_text(value)
        if value:
            return value
    return ""


# Keep the first and last affiliation countries without duplicates.
def first_last_countries(value):
    countries = split_names(value)
    if not countries:
        return []
    return list(dict.fromkeys([countries[0], countries[-1]]))


# Select the papers used in the analysis and the requested observation period.
papers = papers.reset_index(names="paper_id")
papers = papers[
    papers["used_in_analysis"].fillna(False).astype(bool)
    & papers["publication_year"].le(END_YEAR)
    & papers["affiliations_countries"].notna()
].copy()

# Create one row for each distinct lead- or last-author country.
papers["user_country"] = papers["affiliations_countries"].map(
    first_last_countries
)
paper_countries = papers.explode("user_country").dropna(subset=["user_country"])

# Build one row per reviewed resource–holder relationship.
resource_holder_map = source_metadata[[
    "standardized_data_resource_name", "standardized_data_holder_name"
]].copy()
resource_holder_map["standardized_data_holder_name"] = resource_holder_map[
    "standardized_data_holder_name"
].map(split_names)
resource_holder_map = (
    resource_holder_map.explode("standardized_data_holder_name")
    .dropna(subset=["standardized_data_resource_name", "standardized_data_holder_name"])
    .drop_duplicates()
)

# Route 1: holders already stored in the paper-level standardized holder field.
# Notebook 6.1 currently populates this field from both holder and resource rows.
paper_field_holder_uses = paper_countries.assign(
    source_name=paper_countries["standardized_data_holder_names"].map(split_names),
).explode("source_name").dropna(subset=["source_name"])
paper_field_holder_uses = paper_field_holder_uses.assign(
    holder_link_route="paper-level holder field"
)

# Route 2: holders inferred from mentioned resources through the dictionary.
resource_uses = paper_countries.assign(
    standardized_data_resource_name=paper_countries["standardized_data_resource_names"].map(split_names),
).explode("standardized_data_resource_name")
resource_inferred_holder_uses = resource_uses.merge(
    resource_holder_map, on="standardized_data_resource_name", how="inner"
).rename(columns={"standardized_data_holder_name": "source_name"})
resource_inferred_holder_uses = resource_inferred_holder_uses.assign(
    holder_link_route="resource-to-holder mapping"
)

# Union both routes and deduplicate at paper–country–holder level.
holder_use_columns = [
    "paper_id", "doi", "publication_year", "user_country",
    "source_name", "holder_link_route",
]
holder_use_routes = pd.concat(
    [paper_field_holder_uses[holder_use_columns], resource_inferred_holder_uses[holder_use_columns]],
    ignore_index=True,
).drop_duplicates()
holder_uses = (
    holder_use_routes.groupby(
        ["paper_id", "doi", "publication_year", "user_country", "source_name"],
        dropna=False, as_index=False,
    )
    .agg(holder_link_routes=("holder_link_route", lambda values: "; ".join(sorted(set(values)))))
)
holder_uses["source_type"] = "data holder"

# Create one reviewed metadata row per standardized holder. Visualization
# metadata takes precedence, with exact single-holder rows preferred over
# composite semicolon-separated rows and standardized_sources as a fallback.
visual_holder_lookup = visualization_metadata[[
    "standardized_data_holder_name", "standardized_data_holder_type",
    "standardized_data_holder_country_ISO2",
]].copy()
fallback_holder_lookup = source_metadata[[
    "standardized_data_holder_name", "standardized_data_holder_type",
    "standardized_data_holder_country_ISO2",
]].copy()
holder_lookup_parts = []
for priority, lookup in enumerate([visual_holder_lookup, fallback_holder_lookup]):
    lookup["source_name"] = lookup["standardized_data_holder_name"].map(split_names)
    lookup["metadata_specificity"] = lookup["source_name"].map(len)
    lookup = lookup.explode("source_name").dropna(subset=["source_name"])
    lookup["metadata_priority"] = priority
    holder_lookup_parts.append(lookup)
holder_lookup = (
    pd.concat(holder_lookup_parts, ignore_index=True)
    .sort_values(
        ["source_name", "metadata_priority", "metadata_specificity"],
        kind="stable",
    )
    .groupby("source_name", as_index=False, sort=False)
    .agg(
        holder_type=("standardized_data_holder_type", first_text),
        holder_country=("standardized_data_holder_country_ISO2", first_text),
    )
)

# Apply the reviewed decision that Missouri Botanical Garden belongs to the
# hybrid grouping rather than the private-holder grouping.
missouri_mask = holder_lookup["source_name"].eq("Missouri Botanical Garden")
holder_lookup.loc[missouri_mask, ["holder_type", "holder_country"]] = [
    "hybrid",
    "transnational/ambiguous",
]

# Join holder metadata and assign both versions of the analytical categories.
analysis_data = holder_uses.merge(holder_lookup, on="source_name", how="left")
analysis_data["category_differentiated"] = analysis_data.apply(
    assign_differentiated_category, axis=1
)
analysis_data["category"] = analysis_data["category_differentiated"].replace({
    "External US public": "External public",
    "External other public": "External public",
})
analysis_data["country_comparison_category"] = analysis_data.apply(
    assign_country_comparison_category, axis=1
)
analysis_data["country_comparison_category_differentiated"] = analysis_data.apply(
    lambda row: assign_country_comparison_category(row, differentiate_external=True), axis=1
)

# Define a unique observation as one paper-country-source combination.
analysis_data = analysis_data[[
    "doi",
    "publication_year",
    "user_country",
    "paper_id",
    "source_type",
    "source_name",
    "holder_link_routes",
    "holder_type",
    "holder_country",
    "category",
    "category_differentiated",
    "country_comparison_category",
    "country_comparison_category_differentiated",
]].drop_duplicates(["paper_id", "user_country", "source_name"]).reset_index(drop=True)

# Save the complete analysis-level dataset used by the figures.
analysis_data.to_csv(ANALYSIS_DATA_FILE, index=False)
print(f"Paper-level holder-field routes: {len(paper_field_holder_uses):,}")
print(f"Resource-inferred paper–country–holder routes: {len(resource_inferred_holder_uses):,}")
print(f"Combined unique paper–country–holder rows: {len(analysis_data):,}")
print(f"Saved {len(analysis_data):,} rows to {ANALYSIS_DATA_FILE}")


## 2. Create the overall reliance-over-time figures

The first figure combines all external public sources. The additional figure separates US and other external public sources.


In [ ]:
# Create an overall reliance figure for one version of the categories.
def plot_overall_reliance(category_column, category_order, filename, title):
    figure_data = analysis_data[analysis_data[category_column] != "Other / unassigned"]
    shares = (
        figure_data.groupby(["publication_year", category_column])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=category_order, fill_value=0)
    )
    shares = shares.div(shares.sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(11, 6))
    for category in category_order:
        ax.plot(
            shares.index,
            shares[category],
            label=category,
            color=CATEGORY_COLORS[category],
            linewidth=2,
        )

    ax.set(
        title=title,
        xlabel="Publication year",
        ylabel="Share of standardized holder uses (%)",
        ylim=(0, 100),
    )
    ax.grid(axis="y", color="#e2e8f0")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    save_figure(fig, filename)
    plt.show()


# Create the main and differentiated overall figures.
plot_overall_reliance(
    "category",
    CATEGORY_ORDER,
    "human_labelled_source_reliance_over_time",
    "Data-holder reliance over time",
)
plot_overall_reliance(
    "category_differentiated",
    DIFFERENTIATED_CATEGORY_ORDER,
    "human_labelled_source_reliance_over_time_external_public_differentiated",
    "Data-holder reliance over time: external public sources differentiated",
)


## 3. Create the selected-countries figures

Each paper receives a total weight of one, divided equally across the displayed categories it uses. Every EU-hosted institution is treated as a national-public equivalent and as domestic for EU member users, including Germany. In the left column, the stacked height therefore represents the number of papers with at least one displayed category. The right column normalizes those same yearly values to 100% so that changes in composition can be compared independently of publication volume. Non-EU hybrid uses have no displayed country-comparison category, and papers using only such hybrid sources are not shown.


In [ ]:
# Fractionally count each paper across its distinct displayed categories.
def fractional_paper_counts(category_column, category_order):
    paper_categories = analysis_data.dropna(subset=[category_column]).drop_duplicates([
        "doi",
        "user_country",
        "publication_year",
        category_column,
    ])
    paper_keys = ["doi", "user_country", "publication_year"]
    paper_categories["fractional_paper"] = 1 / paper_categories.groupby(paper_keys)[
        category_column
    ].transform("nunique")

    return (
        paper_categories.groupby(["user_country", "publication_year", category_column])[
            "fractional_paper"
        ]
        .sum()
        .unstack(fill_value=0)
        .reindex(columns=category_order, fill_value=0)
    )


# Draw the five country rows as fractional counts and within-year proportions.
def plot_fractional_country_stacks(counts, category_order, filename, title):
    years = range(int(analysis_data["publication_year"].min()), END_YEAR + 1)
    fig, axes = plt.subplots(
        len(SELECTED_COUNTRIES),
        2,
        figsize=(14, 12),
        sharex="col",
        squeeze=False,
    )

    for row, (country_code, country_name) in enumerate(SELECTED_COUNTRIES.items()):
        ax_count, ax_share = axes[row]
        country = counts.loc[country_code].reindex(years, fill_value=0)
        yearly_totals = country.sum(axis=1)
        country_share = country.div(yearly_totals.replace(0, float("nan")), axis=0) * 100
        country_share = country_share.fillna(0)

        ax_count.stackplot(
            country.index,
            *[country[category] for category in category_order],
            labels=category_order,
            colors=[CATEGORY_COLORS[category] for category in category_order],
        )
        ax_share.stackplot(
            country_share.index,
            *[country_share[category] for category in category_order],
            colors=[CATEGORY_COLORS[category] for category in category_order],
        )

        ax_count.set_title(country_name, loc="left", fontweight="bold")
        ax_count.set_ylabel("Papers")
        ax_count.set_ylim(bottom=0)
        ax_share.set_ylabel("Share (%)")
        ax_share.set_ylim(0, 100)
        ax_share.set_yticks([0, 25, 50, 75, 100])

        for ax in (ax_count, ax_share):
            ax.grid(axis="y", color="#e2e8f0")
            ax.spines[["top", "right"]].set_visible(False)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.947),
        ncol=min(len(category_order), 6),
        frameon=False,
        fontsize=12,
        handlelength=2.2,
        handleheight=1.2,
        columnspacing=1.5,
    )
    axes[-1, 0].set_xlabel("Publication year")
    axes[-1, 1].set_xlabel("Publication year")
    fig.suptitle(title, y=0.993, fontsize=18)
    fig.tight_layout(rect=(0, 0.02, 1, 0.93), w_pad=2.4, h_pad=1.2)
    save_figure(fig, filename)
    plt.show()


### 3.1 Main country comparison

This version combines all external national-public and research use.


In [ ]:
# Create the main figure with all external data use combined.
country_counts = fractional_paper_counts(
    "country_comparison_category",
    COUNTRY_COMPARISON_ORDER,
)
plot_fractional_country_stacks(
    country_counts,
    COUNTRY_COMPARISON_ORDER,
    "human_labelled_source_reliance_selected_countries",
    "Data use in selected countries",
)


### 3.2 Country comparison with external use differentiated

This additional version separates external US data use from other external data use.


In [ ]:
# Create the additional figure with US and other external data use separated.
country_counts_differentiated = fractional_paper_counts(
    "country_comparison_category_differentiated",
    COUNTRY_COMPARISON_DIFFERENTIATED_ORDER,
)
plot_fractional_country_stacks(
    country_counts_differentiated,
    COUNTRY_COMPARISON_DIFFERENTIATED_ORDER,
    "human_labelled_source_reliance_selected_countries_external_public_differentiated",
    "Domestic versus external data use in selected countries, 2010–2024",
)


## 4. Compare external public-source use by economic group

This section uses the UN WESP 2026 economic groups and country eligibility threshold from notebook 8.2. Each retained country receives equal weight in its economic-group line and in the overall average.


In [ ]:
# Load country classifications from the overall holder table and retain countries with more than five papers.
ECONOMIC_GROUP_FILE = (
    INPUT_FILE.parent / "processed_data" / "country_relative_use_all_data_holders.csv"
)
MINIMUM_COUNTRY_PAPERS = 5
ECONOMIC_GROUP_ORDER = [
    "Developed economies",
    "Economies in transition",
    "Developing economies",
]
ECONOMIC_GROUP_COLORS = {
    "Developed economies": "#1f77b4",
    "Economies in transition": "#ff7f0e",
    "Developing economies": "#2ca02c",
}

economic_groups = pd.read_csv(
    ECONOMIC_GROUP_FILE,
    usecols=["country", "economic_group", "country_paper_count"],
    keep_default_na=False,
).drop_duplicates()
economic_groups = economic_groups[
    economic_groups["country_paper_count"].gt(MINIMUM_COUNTRY_PAPERS)
    & economic_groups["economic_group"].isin(ECONOMIC_GROUP_ORDER)
][["country", "economic_group"]].drop_duplicates()

# Attach one economic-group classification to each analysis row.
economic_analysis_data = analysis_data.merge(
    economic_groups,
    left_on="user_country",
    right_on="country",
    how="inner",
)


# Calculate country-year shares before averaging, so every country receives equal weight.
def calculate_economic_group_shares(external_categories):
    data = economic_analysis_data[
        economic_analysis_data["category_differentiated"].ne("Other / unassigned")
    ].copy()
    data["external_public_use"] = data["category_differentiated"].isin(
        external_categories
    )

    country_year = (
        data.groupby(["user_country", "economic_group", "publication_year"])
        .agg(
            all_holder_uses=("doi", "size"),
            external_public_uses=("external_public_use", "sum"),
        )
        .reset_index()
    )
    country_year["external_public_share"] = (
        country_year["external_public_uses"] / country_year["all_holder_uses"]
    )

    group_shares = (
        country_year.groupby(["publication_year", "economic_group"])[
            "external_public_share"
        ]
        .mean()
        .unstack()
        .reindex(columns=ECONOMIC_GROUP_ORDER)
    )
    average_share = country_year.groupby("publication_year")[
        "external_public_share"
    ].mean()
    return group_shares, average_share


# Draw and save one economic-group comparison figure.
def plot_economic_group_shares(group_shares, average_share, filename, title):
    fig, ax = plt.subplots(figsize=(10, 6))
    for economic_group in ECONOMIC_GROUP_ORDER:
        ax.plot(
            group_shares.index,
            group_shares[economic_group] * 100,
            label=economic_group,
            color=ECONOMIC_GROUP_COLORS[economic_group],
            linewidth=2.4,
        )

    ax.plot(
        average_share.index,
        average_share * 100,
        label="Average country",
        color="#222222",
        linewidth=2.4,
        linestyle="--",
    )
    ax.set(
        title=title,
        xlabel="Publication year",
        ylabel="External public-source share (%)",
        ylim=(0, 100),
    )
    ax.grid(axis="y", color="#e2e8f0")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    save_figure(fig, filename)
    plt.show()


### 4.1 All external public-source use

This figure combines external US national-public sources and external national-public sources from all other countries.


In [ ]:
# Calculate and plot the share of all holder uses supplied by external public holders.
economic_group_external_shares, average_external_share = calculate_economic_group_shares([
    "External US public",
    "External other public",
])
plot_economic_group_shares(
    economic_group_external_shares,
    average_external_share,
    "economic_group_external_public_share_over_time",
    "External public-source use by economic group",
)


### 4.2 External US public-source use

This figure includes only external national-public sources hosted in the United States.


In [ ]:
# Calculate and plot the share of all holder uses supplied by external US public holders.
economic_group_us_shares, average_us_share = calculate_economic_group_shares([
    "External US public",
])
plot_economic_group_shares(
    economic_group_us_shares,
    average_us_share,
    "economic_group_external_us_public_share_over_time",
    "External US public-source use by economic group",
)


## 5. Compare external public-source use by continent

This section uses the continent classifications and country eligibility threshold from notebook 8.2. Each retained country receives equal weight in its continent line and in the overall average.


In [ ]:
# Reuse continent classifications from the overall holder table and retain countries with more than five papers.
CONTINENT_ORDER = [
    "Africa",
    "Asia",
    "Europe",
    "North America",
    "South America",
    "Oceania",
]
CONTINENT_COLORS = {
    "Africa": "#1f77b4",
    "Asia": "#ff7f0e",
    "Europe": "#2ca02c",
    "North America": "#d62728",
    "South America": "#9467bd",
    "Oceania": "#8c564b",
}

continents = pd.read_csv(
    ECONOMIC_GROUP_FILE,
    usecols=["country", "continent", "country_paper_count"],
    keep_default_na=False,
).drop_duplicates()
continents = continents[
    continents["country_paper_count"].gt(MINIMUM_COUNTRY_PAPERS)
    & continents["continent"].isin(CONTINENT_ORDER)
][["country", "continent"]].drop_duplicates()

# Attach one continent classification to each analysis row.
continent_analysis_data = analysis_data.merge(
    continents,
    left_on="user_country",
    right_on="country",
    how="inner",
)


# Calculate country-year shares before averaging, so every country receives equal weight.
def calculate_continent_shares(external_categories):
    data = continent_analysis_data[
        continent_analysis_data["category_differentiated"].ne("Other / unassigned")
    ].copy()
    data["external_public_use"] = data["category_differentiated"].isin(
        external_categories
    )

    country_year = (
        data.groupby(["user_country", "continent", "publication_year"])
        .agg(
            all_holder_uses=("doi", "size"),
            external_public_uses=("external_public_use", "sum"),
        )
        .reset_index()
    )
    country_year["external_public_share"] = (
        country_year["external_public_uses"] / country_year["all_holder_uses"]
    )

    continent_shares = (
        country_year.groupby(["publication_year", "continent"])[
            "external_public_share"
        ]
        .mean()
        .unstack()
        .reindex(columns=CONTINENT_ORDER)
    )
    average_share = country_year.groupby("publication_year")[
        "external_public_share"
    ].mean()
    return continent_shares, average_share


# Draw and save one continent comparison figure.
def plot_continent_shares(continent_shares, average_share, filename, title):
    fig, ax = plt.subplots(figsize=(10, 6))
    for continent in CONTINENT_ORDER:
        ax.plot(
            continent_shares.index,
            continent_shares[continent] * 100,
            label=continent,
            color=CONTINENT_COLORS[continent],
            linewidth=2.4,
        )

    ax.plot(
        average_share.index,
        average_share * 100,
        label="Average country",
        color="#222222",
        linewidth=2.4,
        linestyle="--",
    )
    ax.set(
        title=title,
        xlabel="Publication year",
        ylabel="External public-source share (%)",
        ylim=(0, 100),
    )
    ax.grid(axis="y", color="#e2e8f0")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    save_figure(fig, filename)
    plt.show()


### 5.1 All external public-source use

This figure combines external US national-public sources and external national-public sources from all other countries.


In [ ]:
# Calculate and plot the share of all holder uses supplied by external public holders.
continent_external_shares, continent_average_external_share = calculate_continent_shares([
    "External US public",
    "External other public",
])
plot_continent_shares(
    continent_external_shares,
    continent_average_external_share,
    "continent_external_public_share_over_time",
    "External public-source use by continent",
)


### 5.2 External US public-source use

This figure includes only external national-public sources hosted in the United States.


In [ ]:
# Calculate and plot the share of all holder uses supplied by external US public holders.
continent_us_shares, continent_average_us_share = calculate_continent_shares([
    "External US public",
])
plot_continent_shares(
    continent_us_shares,
    continent_average_us_share,
    "continent_external_us_public_share_over_time",
    "External US public-source use by continent",
)


## 6. Compare domestic and external data use across countries

Each point represents one country. The axes count distinct papers using domestic or external national-public and research sources between 2010 and 2021. A paper can appear on both axes if it uses both kinds of source. Symmetric logarithmic scales retain countries with zero domestic papers while making the wide range of counts visible.


### 6.1 Paper counts

The count version shows the absolute scale of domestic and external use.


In [ ]:
# Count every paper at most once in each country and data-use category.
country_paper_categories = analysis_data[
    analysis_data["country_comparison_category"].isin([
        "Domestic data use",
        "External data use",
    ])
].drop_duplicates([
    "doi",
    "user_country",
    "country_comparison_category",
])

country_domestic_external = (
    country_paper_categories.groupby([
        "user_country",
        "country_comparison_category",
    ])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Domestic data use", "External data use"], fill_value=0)
)

# Use the same eligible-country set as sections 4 and 5.
eligible_countries = set(economic_groups["country"])
country_domestic_external = country_domestic_external.loc[
    country_domestic_external.index.intersection(eligible_countries)
].copy()
country_domestic_external["total_papers"] = country_domestic_external.sum(axis=1)

# Plot all eligible countries and label those with at least 500 papers across the two axes.
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(
    country_domestic_external["Domestic data use"],
    country_domestic_external["External data use"],
    color="#1f77b4",
    alpha=0.7,
    s=45,
)

for country_code, row in country_domestic_external.iterrows():
    if row["total_papers"] >= 500:
        ax.annotate(
            country_code,
            (row["Domestic data use"], row["External data use"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )

# Add an equality line: countries above it use external sources in more papers than domestic sources.
axis_max = country_domestic_external[[
    "Domestic data use",
    "External data use",
]].to_numpy().max() * 1.1
ax.plot(
    [0, axis_max],
    [0, axis_max],
    color="#7f7f7f",
    linestyle="--",
    linewidth=1,
    label="Equal domestic and external counts",
)

ax.set_xscale("symlog", linthresh=10)
ax.set_yscale("symlog", linthresh=10)
ax.set(
    title="Domestic and external data use across countries",
    xlabel="Papers using domestic national-public or research sources",
    ylabel="Papers using external national-public or research sources",
    xlim=(0, axis_max),
    ylim=(0, axis_max),
)
ax.grid(color="#e2e8f0")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()

save_figure(fig, "country_domestic_external_paper_counts")
plt.show()
